In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

# path data
DATA_DIR = Path("../data/interim")
TRAIN_PATH = DATA_DIR / "train_merged.parquet"

# 02 - EDA: Fitur Transaksi

## Tujuan
- Eksplorasi fitur-fitur pada tabel transaksi
- Memahami pola TransactionDT dan mengekstrak fitur waktu yang berguna
- Menganalisis fitur kartu (card1–card6)
- Melihat distribusi count features (C1–C14) dan V-features (Vesta)

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
print(f"Data loaded: {train.shape}")

In [ ]:
print(f"Columns: {len(train.columns)}")
print(
    f"Transaction data: {len(train[train['isFraud'] == 0]):,} legitimate, {len(train[train['isFraud'] == 1]):,} fraudulent"
)

## TransactionDT - Fitur Waktu


In [ ]:
# fitur waktu transaksi

# konstanta konversi waktu
SECOND_PER_HOUR = 3600
SECOND_PER_DAY = 86400

# ekstrak waktu dari TransactionDT
train["hour"] = (train["TransactionDT"] // SECOND_PER_HOUR) % 24
train["day_of_week"] = (train["TransactionDT"] // SECOND_PER_DAY) % 7
train["day"] = train["TransactionDT"] // SECOND_PER_DAY

print("TransactionDT Statistics :")
print(f"Min : {train['TransactionDT'].min():,} second")
print(f"Max :{train['TransactionDT'].max():,} second")
print(f"Range : {train['TransactionDT'].max() // SECOND_PER_DAY} days")

print("\nExtract Features")
print(f"Hour Range {train['hour'].min()} - {train['hour'].max()}")
print(f"days in data {train['day'].nunique()}")

In [ ]:
# fraud rate per jam

fraud_by_hour = train.groupby("hour").agg({"isFraud": ["sum", "mean", "count"]})
fraud_by_hour.columns = ["fraud_count", "fraud_rate", "total_count"]
fraud_by_hour["fraud_rate"] *= 100

# plot fraud rate per jam
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# jumlah transaksi per jam
axes[0].bar(fraud_by_hour.index, fraud_by_hour["total_count"])
axes[0].set_title("Transaction per Hour")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Count")

# fraud rate per jam
axes[1].bar(fraud_by_hour.index, fraud_by_hour["fraud_rate"], color="#e74c3c")
axes[1].set_title("Fraud Rate per Hour")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].axhline(y=train["isFraud"].mean() * 100, color="black", linestyle="--", label="Overal Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# fraud rate per hari dalam seminggu

fraud_by_dow = train.groupby("day_of_week")["isFraud"].mean() * 100
dow_name = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(7), fraud_by_dow.values)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_name)
ax.set_xlabel("Day of Week")
ax.set_ylabel("Fraud Rate (%)")
ax.axhline(y=train["isFraud"].mean() * 100, color="black", linestyle="--", label="Overall Rate")
ax.legend()
plt.tight_layout()
plt.show()

## Fitur Kartu (card1–card6)

In [ ]:
card_cols = ["card1", "card2", "card3", "card4", "card5", "card6"]

for col in card_cols:
    nunique = train[col].nunique()
    missing = train[col].isnull().sum() / len(train) * 100
    dtype = train[col].dtype
    print(f"{col}: {nunique} unique, {missing:.1f}% missing, type={dtype}")

In [ ]:
print("card4:", train["card4"].dropna().unique())
print("card6:", train["card6"].dropna().unique())

# cek  card1 punya hubungan 1:1 dengan jaringan kartu
for c1 in train["card1"].value_counts().head(5).index:
    nets = train.loc[train["card1"] == c1, "card4"].value_counts(dropna=False)
    print(f"card1={c1}: {dict(nets)}")

In [ ]:
# fraud rate berdasarkan kartu

# card4 dan card6
for col in ["card4", "card6"]:
    fraud_rate = train.groupby(col)["isFraud"].agg(["mean", "count"])
    fraud_rate["mean"] *= 100
    fraud_rate.columns = ["fraud_rate", "count"]

    print(f"\n{col} fraud rate :")
    print(fraud_rate.sort_values("fraud_rate", ascending=False))

    # visualisasi fraud rate per kategori
    fig, ax = plt.subplots(figsize=(10, 5))
    fraud_rate["fraud_rate"].sort_values(ascending=False).plot(kind="bar", ax=ax)
    ax.set_title(f"Fraud by {col}")
    ax.set_ylabel("fraud rate %")
    ax.axhline(y=train["isFraud"].mean() * 100, color="black", linestyle="--", label="overal Rate")
    ax.legend()
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Email

analisis domain email pembeli dan penerima 

In [ ]:
for col in ["P_emaildomain", "R_emaildomain"]:
    top_domains = train[col].value_counts().head(10)

    print(f"\n {col} - Top 10 Domains")
    print(top_domains)

    domain_fraud = train.groupby(col)["isFraud"].mean() * 100
    domain_fraud = domain_fraud.loc[top_domains.index]

    print(f"\n Fraud rate by {col} ")
    print(domain_fraud.sort_values(ascending=False))

In [ ]:
# konversi kategorikal ke string untuk perbandingan
train["email_match"] = (
    train["P_emaildomain"].astype(str) == train["R_emaildomain"].astype(str)
).fillna(False)

email_match_fraud = train.groupby("email_match")["isFraud"].mean() * 100
print(email_match_fraud)

## Fitur Count (C1–C14)

In [ ]:
c_cols = [f"C{i}" for i in range(1, 15)]

print("Count Feature")
print(train[c_cols].describe())

In [ ]:
# korelasi C-features dengan fraud
c_corr = train[c_cols + ["isFraud"]].corr()["isFraud"].drop("isFraud").sort_values()

print("c features correlation with isfraud :")
print(c_corr)

# visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
c_corr.plot(kind="barh", ax=ax)
ax.set_title("c features correlation with isfraud")
ax.set_xlabel("correlation")
ax.axvline(x=0, color="black", linestyle="--")
plt.tight_layout()
plt.show()

## V-Features (Vesta)


In [ ]:
# V-features (Vesta)
v_cols = [col for col in train.columns if col.startswith("V") and col[1:].isdigit()]

print(f"vesta features : {len(v_cols)} columns")

# hitung persentase missing per V-feature
v_missing = train[v_cols].isnull().sum() / len(train) * 100

print(f"\nMissing Values in V Features:")
print(f" 0% missing: {(v_missing == 0).sum()} columns")
print(f">0% missing: {(v_missing > 0).sum()} columns")
print(f">50% missing: {(v_missing > 50).sum()} columns")
print(f">90% missing: {(v_missing > 90).sum()} columns")

In [ ]:
# top V-features berdasarkan korelasi
# hitung korelasi hanya missing < 50%
v_cols_clean = [c for c in v_cols if v_missing[c] < 50]

v_corr = train[v_cols_clean + ["isFraud"]].corr()["isFraud"].drop("isFraud").sort_values()

# top korelasi positif dan negatif
top_positive = v_corr.nlargest(10)
top_negative = v_corr.nsmallest(10)

print("top 10 positive correlation")
print(top_positive)

print("top 10 negative correlation")
print(top_negative)